# Bailian Agent（SDK）測試：取得回覆與 `doc_references` / chunk

目的：用 **dashscope** 官方 SDK 單獨呼叫你的 Chat Agent，檢查 **非串流 / 串流** 回傳裡是否含知識庫引用。

前置：
- 在專案 `backend/.env` 設定 `BAILIAN_API_KEY`（或 `DASHSCOPE_API_KEY`）與 `BAILIAN_APP_ID_CHAT`
- 百煉控制台已開 **Show Source** 並 **發布**應用
- 本筆記本建議 **Kernel 使用 backend 虛擬環境**（已 `pip install dashscope`）

若 SDK 與 HTTP 行為不一致，可對照 `backend/scripts/test_bailian_doc_refs.py`（純 HTTP）。

In [2]:
# 安裝 SDK（每個 venv 只需一次）
%pip install dashscope -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import os
from http import HTTPStatus
from pathlib import Path

from dotenv import load_dotenv

# 筆記本在 backend/notebooks/ → backend 目錄為父級
def _find_backend_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "backend" / ".env").is_file():
            return p / "backend"
        if (p / ".env").is_file() and (p / "app").is_dir():
            return p
        p = p.parent
    return Path.cwd()

BACKEND_ROOT = _find_backend_root()
load_dotenv(BACKEND_ROOT / ".env")

API_KEY = os.getenv("BAILIAN_API_KEY") or os.getenv("DASHSCOPE_API_KEY")
APP_ID = os.getenv("BAILIAN_APP_ID_CHAT", "").strip()
# 與後端 bailian_service 一致；國際版可改為 https://dashscope-intl.aliyuncs.com/api/v1
BASE_URL = os.getenv("BAILIAN_BASE", "https://dashscope.aliyuncs.com/api/v1")

if not API_KEY or not APP_ID:
    raise RuntimeError("請在 backend/.env 設定 BAILIAN_API_KEY 與 BAILIAN_APP_ID_CHAT")

import dashscope

dashscope.api_key = API_KEY
dashscope.base_http_api_url = BASE_URL.rstrip("/")


def _output_to_dict(out):
    """Convert Application output to a plain dict.
    dashscope uses DictMixin: hasattr(out, 'to_dict') triggers __getitem__('to_dict') → KeyError.
    """
    if out is None:
        return {}
    if isinstance(out, dict):
        return dict(out)
    try:
        return dict(out)
    except (TypeError, ValueError):
        pass
    try:
        return {k: out[k] for k in out.keys()}
    except Exception:
        pass
    return {"text": getattr(out, "text", str(out))}


print("BASE:", dashscope.base_http_api_url)
print("APP_ID:", APP_ID[:8] + "...")

BASE: https://dashscope.aliyuncs.com/api/v1
APP_ID: fdf91438...


## 1）非串流 `Application.call`（最容易看到完整 `output`）

`output` 裡可能出現 **`doc_references` 這個 key，但值為 `null`**（JSON 的 null）：代表本輪**沒有回傳結構化 chunk 陣列**，並不等於「沒開 Show Source」。此時引用常改以正文裡的 **`[^n]: [標題](url)` 腳註** 呈現，請看下方「footnote links in text」。

In [14]:
import re

from dashscope import Application

QUESTION = (
    "請問香港科技大學本科生宿舍 Hall 7 本地生雙人房每年的住宿費大約是多少港元？請依知識庫簡短回答。"
)

resp = Application.call(api_key=API_KEY, app_id=APP_ID, prompt=QUESTION)

print("status_code:", resp.status_code)
print("request_id:", getattr(resp, "request_id", None))

if resp.status_code != HTTPStatus.OK:
    print("error:", getattr(resp, "message", resp))
else:
    od = _output_to_dict(resp.output)
    print("\n--- output keys ---")
    print(list(od.keys()))
    refs = od.get("doc_references")
    print("doc_references raw:", repr(refs))
    if "doc_references" not in od:
        print("doc_references: <key absent>")
    elif refs is None:
        print(
            "doc_references: null — API 有欄位但無結構化列表；"
            "引用可能在正文 [^n] 腳註（見下方）"
        )
    elif isinstance(refs, list) and len(refs) == 0:
        print("doc_references: [] (empty list)")
    else:
        print("doc_references count:", len(refs))
        print(json.dumps(refs[:3], ensure_ascii=False, indent=2))

    text = od.get("text") or ""
    foot_urls = re.findall(r"\[\^\d+\]:\s*\[([^\]]*)\]\((https?://[^)\s]+)\)", text)
    print("\n--- footnote links parsed from text ---")
    if not foot_urls:
        print("(none)")
    else:
        for title, url in foot_urls:
            print(f"- {title!s}")
            print(f"  {url}")

    print("\n--- text preview (first 500 chars) ---")
    print(text[:500])

status_code: 200
request_id: ddbc21df-88de-978a-bc89-fefdbe32831c

--- output keys ---
['text', 'finish_reason', 'session_id', 'thoughts', 'doc_references', 'workflow_message']
doc_references raw: None
doc_references: null — API 有欄位但無結構化列表；引用可能在正文 [^n] 腳註（見下方）

--- footnote links parsed from text ---
- HKUST Undergraduate Hall Charges & Room Availability Matrix
  https://bailian-datahub-data-prod.oss-cn-beijing.aliyuncs.com/13396274/multimodal/docJson/hall_charges%26%20Room%20Availability%20Matrix%20_1773068265391.json?Expires=1773327469&OSSAccessKeyId=LTAI5tKzNnKPFwCJSCpxx51h&Signature=FxjCoY13Uyo1Ly3L%2BPdUceg54zk%3D
- UG Hall VII - Bedroom Facilities
  https://bailian-datahub-data-prod.oss-cn-beijing.aliyuncs.com/13396274/multimodal/docJson/hall_facilities_1772642285738.json?Expires=1772901501&OSSAccessKeyId=LTAI5tKzNnKPFwCJSCpxx51h&Signature=EUeJuI%2BCvT8sWu%2B6%2FExbjTzpdYI%3D
- UG Hall VII - CSK Hall Bedroom
  https://bailian-datahub-data-prod.oss-cn-beijing.aliyuncs.com/13396274

## 2）串流 `Application.call(..., stream=True)`

最後一個 chunk 可能才帶 `doc_references`；若沒有，與 HTTP SSE 行為類似，屬常見現象。

In [15]:
try:
    gen = Application.call(api_key=API_KEY, app_id=APP_ID, prompt=QUESTION, stream=True)
except TypeError as e:
    print("SDK 不支援 stream=True 或參數不同:", e)
    gen = None

last = None
if gen is not None:
    for chunk in gen:
        last = chunk
        if chunk.status_code != HTTPStatus.OK:
            print("stream error:", chunk.message)
            break

if last is not None and last.status_code == HTTPStatus.OK:
    od = _output_to_dict(last.output)
    print("last chunk output keys:", list(od.keys()))
    dr = od.get("doc_references")
    print("doc_references raw:", repr(dr))
    if dr is None and "doc_references" in od:
        print("(null — 同非串流：可能僅有正文腳註連結)")
    else:
        print("doc_references:", dr)

last chunk output keys: ['text', 'finish_reason', 'session_id', 'thoughts', 'doc_references', 'workflow_message']
doc_references raw: None
(null — 同非串流：可能僅有正文腳註連結)


## 3）可選：用 `messages` 多輪（與後端一致時）

若你的後端使用 `input.messages` 而非單一 `prompt`，SDK 需支援對應參數。請以 [百煉 Application API](https://help.aliyun.com/zh/model-studio/agent-and-workflow-application-api-reference) 為準；若 `Application.call` 僅支援 `prompt`，則多輪請改走 HTTP（`test_bailian_doc_refs.py` 可自行改 payload）。